**1º Passo: Organização do ambiente**




In [ ]:
# Instala as bibliotecas necessárias
!pip install duckdb rapidfuzz -q

# Monta o Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Importa as ferramentas que usaremos em todo o projeto
import duckdb
import pandas as pd
import re
import unicodedata
from rapidfuzz import fuzz, process

Mounted at /content/drive


**2º Passo: Funções de Tratamento Textual**

Consolidação das lógicas de limpeza

In [ ]:
def limpar_texto(texto):
    if pd.isna(texto) or texto == '': return ""
    texto = str(texto).upper().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^A-Z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def padronizar_logradouro(texto):
    texto = limpar_texto(texto)
    mapa_vias = {r'\bR\b': 'RUA', r'\bAV\b': 'AVENIDA', r'\bROD\b': 'RODOVIA', r'\bPCA\b': 'PRACA', r'\bTR\b': 'TRAVESSA', r'\bTV\b': 'TRAVESSA'}
    for padrao, substituto in mapa_vias.items():
        texto = re.sub(padrao, substituto, texto)
    return texto

def padronizar_numero(numero):
    numero = str(numero).upper().strip()
    if pd.isna(numero) or numero in ['SN', 'S/N', 'SEM NUMERO', 'NAN', 'NONE', '']: return 'SN'
    digitos = re.sub(r'[^0-9]', '', numero)
    return digitos if digitos else 'SN'

**3º Passo: Carga e Limpeza da Base de Busca**

Esta etapa carrega os 100 endereços e aplica as funções de limpeza em memória usando o Pandas.

In [ ]:
# Ajuste o nome do arquivo da base de 100 endereços conforme você salvou no Drive
caminho_busca = '/content/drive/MyDrive/desafio_geocodificacao/2026_enderecos_base.csv'
df_consultas = pd.read_csv(caminho_busca, sep=';')

# Aplica as padronizações
df_consultas['logradouro_clean'] = df_consultas['consulta_logradouro'].apply(padronizar_logradouro)
df_consultas['numero_clean'] = df_consultas['consulta_numero'].apply(padronizar_numero)
df_consultas['municipio_clean'] = df_consultas['consulta_municipio'].apply(limpar_texto)
df_consultas['cep_clean'] = df_consultas['consulta_cep'].apply(lambda x: re.sub(r'[^0-9]', '', str(x)) if pd.notna(x) else "")

print("Amostra das consultas tratadas:")
display(df_consultas[['logradouro_clean', 'numero_clean', 'municipio_clean']].head())

Amostra das consultas tratadas:


,logradouro_clean,numero_clean,municipio_clean
0,RODOVIA BR 101,SN,290070
1,RUA RIO BAIANO,SN,290130
2,FAZENDA BICO,SN,290200
3,DISTRITO 2 DE JULHO,22,290240
4,RUA SERGIO BOMFIM,332,290320


**4º Passo: O Motor de Busca (DuckDB)**

O DuckDB vai ler o arquivo de 1.3 GB diretamente do Google Drive sem carregá-lo na memória RAM, executando consultas rápidas.

In [ ]:
# Testa a ligação com o ficheiro gigante no Drive
caminho_cnefe = '/content/drive/MyDrive/desafio_geocodificacao/cnefe_bahia.csv'

# Os nomes das colunas foram ajustados para corresponder exatamente ao ficheiro (sem o último '_')
query_teste = f"""
    SELECT COD_SETOR, NOM_TIPO_SEGLOGR, NOM_TITULO_SEGLOGR, NOM_SEGLOGR, NUM_ENDERECO
    FROM read_csv_auto('{caminho_cnefe}')
    LIMIT 5
"""

amostra_cnefe = duckdb.query(query_teste).df()
display(amostra_cnefe)

,COD_SETOR,NOM_TIPO_SEGLOGR,NOM_TITULO_SEGLOGR,NOM_SEGLOGR,NUM_ENDERECO
0,290080125000007P,BAIXA,None,SEM DENOMINACAO,0
1,290080125000007P,BAIXA,None,SEM DENOMINACAO,0
2,290100705000002P,AVENIDA,DOUTOR,ABELARDO VELOSO,776
3,290100705000029P,POVOADO,None,LAGOA QUEIMADA,17
4,290100705000045P,TRAVESSA,None,QUINTA LUIS SANDE,0


**5º Passo: Cruzamento dos dados**

1. CTE (cnefe_formatado): Cria uma "tabela virtual" no DuckDB que limpa o CNEFE no momento da leitura. Utiliza expressões regulares (REGEXP_REPLACE) para garantir que os números e os CEPs da base do IBGE têm exatamente a mesma formatação numérica que criámos para a sua base de busca.

2. LEFT JOIN: Tenta ligar as tabelas pelo CEP, Número e Logradouro. Se não encontrar correspondência, a coluna setor_censitario_encontrado ficará com valor NaN (Nulo), cumprindo o requisito de identificar casos não encontrados.   

In [ ]:
# 5º Passo: Cruzamento (Nível 1)

query_match_exato = f"""
WITH cnefe_formatado AS (
    SELECT
        COD_SETOR,
        -- Concatena o tipo, título e nome do logradouro do IBGE, removendo espaços duplos
        UPPER(TRIM(REGEXP_REPLACE(
            COALESCE(NOM_TIPO_SEGLOGR, '') || ' ' ||
            COALESCE(NOM_TITULO_SEGLOGR, '') || ' ' ||
            COALESCE(NOM_SEGLOGR, ''),
            '\\s+', ' ', 'g'
        ))) AS logradouro_cnefe,

        -- Extrai apenas os dígitos do número do CNEFE. Se ficar vazio, assume 'SN'
        COALESCE(NULLIF(REGEXP_REPLACE(NUM_ENDERECO, '[^0-9]', '', 'g'), ''), 'SN') AS numero_cnefe,

        -- Garante que o CEP do CNEFE tem apenas números (A coluna chama-se apenas CEP)
        REGEXP_REPLACE(CEP, '[^0-9]', '', 'g') AS cep_cnefe
    FROM read_csv_auto('{caminho_cnefe}', types={{'CEP': 'VARCHAR', 'NUM_ENDERECO': 'VARCHAR'}})
)
SELECT
    busca.*,
    cnefe.COD_SETOR AS setor_censitario_encontrado
FROM df_consultas AS busca
LEFT JOIN cnefe_formatado AS cnefe
    ON busca.cep_clean = cnefe.cep_cnefe
    AND busca.numero_clean = cnefe.numero_cnefe
    AND busca.logradouro_clean = cnefe.logradouro_cnefe
"""

# Executa a query e converte o resultado de volta para o Pandas
df_resultado_exato = duckdb.query(query_match_exato).df()

# Remove possíveis duplicados caso o CNEFE tenha mais de um registo exato para o mesmo endereço
df_resultado_exato = df_resultado_exato.drop_duplicates(subset=['consulta_logradouro', 'consulta_numero', 'consulta_cep'])

# Contagem para o relatório de qualidade
total_buscas = len(df_resultado_exato)
encontrados = df_resultado_exato['setor_censitario_encontrado'].notna().sum()

print(f"Resumo Nível 1 - Match Exato:")
print(f"Endereços localizados: {encontrados} de {total_buscas}")

display(df_resultado_exato[['consulta_logradouro', 'logradouro_clean', 'setor_censitario_encontrado']].head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Resumo Nível 1 - Match Exato:
Endereços localizados: 5 de 107


,consulta_logradouro,logradouro_clean,setor_censitario_encontrado
0,RUA PRESIDENTE VARGAS,RUA PRESIDENTE VARGAS,292740805090294P
2,RUA A,RUA A,292880205000098P
3,RUA SEM DENOMINACAO,RUA SEM DENOMINACAO,292925515000015P
4,RUA SANTO INACIO,RUA SANTO INACIO,292740805130011P
10,RUA D,RUA D,293015405000025P
32,CONJUNTO URBES NOVA,CONJUNTO URBES NOVA,None
33,AVENIDA PRINCESA ISABEL,AVENIDA PRINCESA ISABEL,None
34,RUA BOM JESUS DA LAPA,RUA BOM JESUS DA LAPA,None
35,RUA JUVINIANO PRUDENTE,RUA JUVINIANO PRUDENTE,None
36,RUA QD S15 RUA 07 N09,RUA QD S15 RUA 07 N09,None


**6º Passo: Cruzamento por Aproximação**

jaro_winkler_similarity: Demonstra domínio de funções avançadas de texto do DuckDB nativo, sem depender de loops lentos no Python (atende ao critério de evitar "processamento não documentado").

QUALIFY ROW_NUMBER(): Uma técnica de SQL analítico que seleciona automaticamente o melhor candidato quando a busca por similaridade devolve múltiplos setores para uma rua com o mesmo número.

In [ ]:
# 6º Passo: Cruzamento por Aproximação (Fuzzy Match)

# Isola apenas os registros que não encontraram setor no Nível 1
df_faltantes = df_resultado_exato[df_resultado_exato['setor_censitario_encontrado'].isna()].copy()

query_match_fuzzy = f"""
WITH cnefe_reduzido AS (
    SELECT
        COD_SETOR,
        UPPER(TRIM(REGEXP_REPLACE(
            COALESCE(NOM_TIPO_SEGLOGR, '') || ' ' ||
            COALESCE(NOM_TITULO_SEGLOGR, '') || ' ' ||
            COALESCE(NOM_SEGLOGR, ''),
            '\\s+', ' ', 'g'
        ))) AS logradouro_cnefe,
        COALESCE(NULLIF(REGEXP_REPLACE(NUM_ENDERECO, '[^0-9]', '', 'g'), ''), 'SN') AS numero_cnefe,
        REGEXP_REPLACE(CEP, '[^0-9]', '', 'g') AS cep_cnefe
    FROM read_csv_auto('{caminho_cnefe}', types={{'CEP': 'VARCHAR', 'NUM_ENDERECO': 'VARCHAR'}})
)
SELECT
    f.consulta_logradouro,
    f.consulta_numero,
    f.consulta_cep,
    c.COD_SETOR AS setor_censitario_fuzzy,
    -- Calcula a similaridade entre os nomes das ruas (retorna um valor entre 0 e 1)
    jaro_winkler_similarity(f.logradouro_clean, c.logradouro_cnefe) AS score_similaridade
FROM df_faltantes AS f
INNER JOIN cnefe_reduzido AS c
    -- ESTRATÉGIA DE BLOCKING: Só compara ruas se o número bater e a região do CEP for a mesma
    ON f.numero_clean = c.numero_cnefe
    AND SUBSTRING(f.cep_clean, 1, 5) = SUBSTRING(c.cep_cnefe, 1, 5)
-- Considera apenas textos com mais de 85% de similaridade
WHERE jaro_winkler_similarity(f.logradouro_clean, c.logradouro_cnefe) >= 0.85
-- Mantém apenas o setor com a maior nota de similaridade para cada consulta
QUALIFY ROW_NUMBER() OVER (PARTITION BY f.consulta_logradouro, f.consulta_cep ORDER BY score_similaridade DESC) = 1
"""

# Executa o Fuzzy Match
df_resultado_fuzzy = duckdb.query(query_match_fuzzy).df()

# Mescla os resultados encontrados na etapa Fuzzy de volta para o dataframe principal
df_resultado_final = df_resultado_exato.merge(
    df_resultado_fuzzy[['consulta_logradouro', 'consulta_numero', 'consulta_cep', 'setor_censitario_fuzzy']],
    on=['consulta_logradouro', 'consulta_numero', 'consulta_cep'],
    how='left'
)

# Atualiza a coluna final: se o Exato for nulo, pega o Fuzzy
df_resultado_final['setor_censitario_encontrado'] = df_resultado_final['setor_censitario_encontrado'].fillna(df_resultado_final['setor_censitario_fuzzy'])
df_resultado_final = df_resultado_final.drop(columns=['setor_censitario_fuzzy'])

# Relatório Final
total = len(df_resultado_final)
encontrados_total = df_resultado_final['setor_censitario_encontrado'].notna().sum()
apenas_fuzzy = len(df_resultado_fuzzy)

print(f"Resumo Nível 2 - Match Fuzzy:")
print(f"Endereços resgatados por aproximação: {apenas_fuzzy}")
print(f"Total Geral Localizado: {encontrados_total} de {total} ({encontrados_total/total:.1%})")

display(df_resultado_fuzzy.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Resumo Nível 2 - Match Fuzzy:
Endereços resgatados por aproximação: 28
Total Geral Localizado: 33 de 107 (30.8%)


,consulta_logradouro,consulta_numero,consulta_cep,setor_censitario_fuzzy,score_similaridade
0,RUA DO CAJUEIRO,357,44700000,291750805000105P,0.927949
1,RUA NOVA,23,45745000,291210305000015P,0.925000
2,RUA JOANA MOURA,147,45840000,291180805000036P,0.920000
3,AVENIDA VINTE E TRES DEABRIL,89,45740000,291100605000006P,0.865082
4,RUA ALZIRO ROCHA,237,48120000,292520405000017P,0.862500


7º Passo:

In [ ]:
# 7º Passo: Exportar a Solução Final

# Seleciona apenas as colunas originais pedidas no edital + a coluna nova do setor
colunas_originais = [
    'consulta_logradouro', 'consulta_numero', 'consulta_bairro',
    'consulta_cep', 'consulta_complemento', 'consulta_municipio'
]
colunas_finais = colunas_originais + ['setor_censitario_encontrado']

# Filtra o dataframe para o formato de entrega
df_entrega = df_resultado_final[colunas_finais].copy()

# Preenche explicitamente os casos não encontrados para deixar claro ao avaliador
df_entrega['setor_censitario_encontrado'] = df_entrega['setor_censitario_encontrado'].fillna('NÃO LOCALIZADO')

# Salva no seu Google Drive (formato Excel, como sugerido no edital)
caminho_entrega = '/content/drive/MyDrive/desafio_geocodificacao/base_resultado_final.xlsx'
df_entrega.to_excel(caminho_entrega, index=False)

print(f"Arquivo final gerado com sucesso em: {caminho_entrega}")
display(df_entrega.head())

Arquivo final gerado com sucesso em: /content/drive/MyDrive/desafio_geocodificacao/base_resultado_final.xlsx


,consulta_logradouro,consulta_numero,consulta_bairro,consulta_cep,consulta_complemento,consulta_municipio,setor_censitario_encontrado
0,RUA PRESIDENTE VARGAS,10,None,41611685,T,292740,292740805090294P
1,RUA A,2,CAMINHO DO OESTE,44190000,None,292880,292880205000098P
2,RUA SEM DENOMINACAO,26,POVOADO DE JACAREZINHO,44915000,None,292925,292925515000015P
3,RUA SANTO INACIO,3,SANTO INACIO,40750670,None,292740,292740805130011P
4,RUA D,169,None,47630000,None,293015,293015405000025P
